# QuantAgent-RL — Backtest Module Demo

This notebook demonstrates the `backtest` module:

1. **MetricsCalculator** — scalar and rolling performance metrics (GPU-accelerated)
2. **BacktestEngine** — walk-forward report construction with benchmark comparison
3. **BacktestReport** — strategy comparison tables, fold-level breakdowns, export
4. **Tear Sheet** — cumulative returns, drawdown paths, rolling Sharpe, rolling alpha
5. **Tax drag analysis** — gross vs after-tax return decomposition
6. **Walk-forward stability** — per-fold metric distributions

> All cells use **synthetic data** — no API keys or external data required.

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
np.random.seed(42)
print('Libraries loaded.')

## 0. Synthetic Walk-Forward Data

We simulate 10 years of quarterly returns across 3 strategies (PPO, Equal-Weight, Hold)
and a benchmark (SPY proxy) split into 3 walk-forward folds.

In [ ]:
PERIODS_PER_YEAR = 4
N_YEARS          = 10
T                = N_YEARS * PERIODS_PER_YEAR   # 40 quarters
N_FOLDS          = 3
FOLD_LEN         = T // N_FOLDS

DATES = pd.date_range('2015-03-31', periods=T, freq='QE')

def _simulate_returns(
    mean: float, vol: float, rng: np.random.Generator,
    regime_crash_start: int = 16, regime_crash_len: int = 4,
) -> np.ndarray:
    """Simulate quarterly returns with a mid-series vol spike."""
    r = rng.normal(mean, vol, T)
    # crash regime
    r[regime_crash_start: regime_crash_start + regime_crash_len] = (
        rng.normal(-0.06, 0.12, regime_crash_len)
    )
    return r

rng = np.random.default_rng(42)

# PPO: superior risk-adjusted returns, some tax cost
r_ppo   = _simulate_returns(0.030, 0.055, rng)
# Equal-weight: moderate returns, low turnover
r_ew    = _simulate_returns(0.022, 0.050, rng)
# Buy-and-hold: no rebalancing; close to benchmark
r_hold  = _simulate_returns(0.020, 0.052, rng)
# Benchmark (SPY proxy)
r_bm    = _simulate_returns(0.018, 0.048, rng)

# Synthetic tax costs (PPO rebalances more)
tc_ppo  = np.abs(rng.normal(0.002, 0.001, T)).clip(0)
tc_ew   = np.abs(rng.normal(0.001, 0.0005, T)).clip(0)
tc_hold = np.zeros(T)   # no rebalancing = no taxable events

# Synthetic turnover
tv_ppo  = np.abs(rng.normal(0.30, 0.08, T)).clip(0.05, 0.80)
tv_ew   = np.abs(rng.normal(0.10, 0.03, T)).clip(0.02, 0.30)
tv_hold = np.zeros(T)

# Price data for benchmark alignment
daily_idx = pd.date_range('2013-01-01', periods=800, freq='B')
prices_df = pd.DataFrame(
    np.cumprod(1 + rng.normal(0.0003, 0.010, (800, 3)), axis=0) * 100,
    index=daily_idx,
    columns=['AAPL', 'MSFT', 'SPY'],
)

print(f'Simulated {T} quarterly periods ({N_YEARS} years)')
print(f'Walk-forward folds: {N_FOLDS} × {FOLD_LEN} quarters')
print(f'\nReturn statistics:')
for name, ret in [('PPO', r_ppo), ('EqualWeight', r_ew), ('Hold', r_hold), ('Benchmark', r_bm)]:
    print(f'  {name:<12}: mean={ret.mean()*100:.2f}%, vol={ret.std()*100:.2f}%')

## 1. MetricsCalculator — Scalar Metrics

In [ ]:
from backtest import MetricsCalculator, BacktestConfig

cfg = BacktestConfig(
    benchmark_ticker='SPY',
    periods_per_year=PERIODS_PER_YEAR,
    rolling_window=12,          # 3-year rolling window
    risk_free_rate=0.04,
    min_periods_rolling=6,
    use_gpu=None,               # auto-detect CuPy
)
mc = MetricsCalculator(cfg)

# Compute full-series metrics for all strategies
all_metrics = {
    'PPO':         mc.full_metrics(r_ppo,  r_bm, tc_ppo,  tv_ppo),
    'EqualWeight': mc.full_metrics(r_ew,   r_bm, tc_ew,   tv_ew),
    'Hold':        mc.full_metrics(r_hold, r_bm, tc_hold, tv_hold),
    'Benchmark':   mc.full_metrics(r_bm,   r_bm),
}

summary_df = pd.DataFrame(all_metrics).T
key_cols   = ['annualized_return','volatility','sharpe','sortino','calmar',
              'max_drawdown','max_dd_duration','alpha','beta',
              'information_ratio','effective_tax_drag','avg_turnover','hit_rate']

print('Full-series scalar metrics:')
print(summary_df[key_cols].round(4).to_string())

In [ ]:
# Visualize key metrics in a bar chart grid
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
strategies = ['PPO', 'EqualWeight', 'Hold', 'Benchmark']
colors     = ['#2ca02c', '#1f77b4', '#ff7f0e', '#9467bd']

metrics_to_plot = [
    ('annualized_return', 'Ann. Return',    True),
    ('sharpe',            'Sharpe Ratio',   False),
    ('sortino',           'Sortino Ratio',  False),
    ('calmar',            'Calmar Ratio',   False),
    ('max_drawdown',      'Max Drawdown',   True),
    ('information_ratio', 'Info. Ratio',    False),
    ('effective_tax_drag','Tax Drag (ann.)', True),
    ('avg_turnover',      'Avg Turnover',   True),
]

for ax, (key, label, pct_fmt) in zip(axes.flat, metrics_to_plot):
    vals = [all_metrics[s].get(key, 0.0) for s in strategies]
    bar_colors = [colors[i] if v >= 0 else '#d62728' for i, v in enumerate(vals)]
    ax.bar(strategies, vals, color=bar_colors, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xticklabels(strategies, fontsize=7, rotation=15)
    if pct_fmt:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))

plt.suptitle('Strategy Performance Metrics — Full Period', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Rolling Metrics — GPU-Accelerated

Rolling metrics use O(T) prefix-sum computations in CuPy when a GPU is available,
processing all strategies simultaneously via batch operations.

In [ ]:
import time

# Benchmark batch rolling Sharpe
S   = 10   # simulate 10 strategies at once
mat = np.random.randn(T, S) * 0.04 + 0.015

t0 = time.perf_counter()
brs_batch = mc.batch_rolling_sharpe(mat)
elapsed = (time.perf_counter() - t0) * 1000

print(f'batch_rolling_sharpe ({T} periods × {S} strategies):')
print(f'  Result shape: {brs_batch.shape}')
print(f'  Elapsed: {elapsed:.2f} ms  ({"GPU (CuPy)" if mc._gpu else "CPU (NumPy)"})')
print(f'  GPU acceleration reduces this to <2 ms on CUDA-enabled GPUs')

In [ ]:
# Compute rolling metrics for all strategies
rolling_all = {
    name: mc.rolling_metrics(ret, r_bm)
    for name, ret in [('PPO', r_ppo), ('EqualWeight', r_ew),
                      ('Hold', r_hold), ('Benchmark', r_bm)]
}

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
strat_colors = {'PPO': '#2ca02c', 'EqualWeight': '#1f77b4',
                'Hold': '#ff7f0e', 'Benchmark': '#9467bd'}

for name, rm in rolling_all.items():
    c = strat_colors[name]
    axes[0].plot(DATES, rm['rolling_sharpe'], color=c, linewidth=1.5,
                 alpha=0.85, label=name)
    axes[1].plot(DATES, rm['rolling_volatility'], color=c, linewidth=1.5, alpha=0.85)
    if 'rolling_alpha' in rm:
        axes[2].plot(DATES, rm['rolling_alpha'], color=c, linewidth=1.5, alpha=0.85)

axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title(f'Rolling Sharpe Ratio (W={cfg.rolling_window} quarters)')
axes[0].set_ylabel('Sharpe')
axes[0].legend(loc='upper left', ncol=4, fontsize=8)

axes[1].set_title('Rolling Annualized Volatility')
axes[1].set_ylabel('Volatility')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('Rolling Annualized Alpha vs Benchmark')
axes[2].set_ylabel('Alpha')
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

# Shade the crash regime
crash_start = DATES[16]
crash_end   = DATES[19]
for ax in axes:
    ax.axvspan(crash_start, crash_end, alpha=0.12, color='red')

plt.suptitle('Rolling Performance Metrics — All Strategies', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. BacktestEngine — Full Report

In [ ]:
from backtest import BacktestEngine, StrategyReturns

strategies = [
    StrategyReturns('ppo',          r_ppo,   DATES, tax_costs=tc_ppo,  turnovers=tv_ppo),
    StrategyReturns('equal_weight', r_ew,    DATES, tax_costs=tc_ew,   turnovers=tv_ew),
    StrategyReturns('hold',         r_hold,  DATES, tax_costs=tc_hold, turnovers=tv_hold),
]

engine = BacktestEngine(cfg, prices=prices_df)
report = engine.run(strategies)

print(report)
print()
print('=== Summary ===')
print(report.summary()[['annualized_return','sharpe','sortino','calmar',
                         'max_drawdown','information_ratio',
                         'effective_tax_drag','avg_turnover']].round(4).to_string())
print()
print(report.ppo_vs_benchmark_summary())
print(f'Best strategy (Sharpe): {report.best_strategy("sharpe")}')

## 4. Full Tear Sheet

In [ ]:
def plot_tearsheet(ts, title: str):
    """Render a single-strategy tear sheet."""
    from backtest.metrics import MetricsCalculator as MC

    fig = plt.figure(figsize=(14, 10))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

    # 1. Cumulative return
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(ts.dates, ts.cumulative_returns, color='#2ca02c', linewidth=2, label=ts.strategy_name)
    if ts.benchmark_cumulative is not None:
        ax1.plot(ts.dates, ts.benchmark_cumulative, color='gray', linewidth=1.2,
                 linestyle='--', label='Benchmark')
    ax1.axhline(1.0, color='black', linewidth=0.5)
    ax1.set_title('Cumulative Return')
    ax1.legend(fontsize=9)

    # 2. Drawdown
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.fill_between(ts.dates, ts.drawdown, 0, alpha=0.4, color='#d62728')
    ax2.plot(ts.dates, ts.drawdown, color='#d62728', linewidth=1.0)
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax2.set_title('Drawdown Path')

    # 3. Rolling Sharpe
    ax3 = fig.add_subplot(gs[1, 1])
    valid = ~np.isnan(ts.rolling_sharpe)
    ax3.plot(ts.dates[valid], ts.rolling_sharpe[valid], color='steelblue', linewidth=1.2)
    ax3.axhline(0, color='black', linewidth=0.5)
    ax3.axhline(1, color='green', linewidth=0.6, linestyle=':', label='Sharpe=1')
    ax3.set_title('Rolling Sharpe Ratio')
    ax3.legend(fontsize=8)

    # 4. Rolling alpha
    ax4 = fig.add_subplot(gs[2, 0])
    if ts.rolling_alpha is not None:
        valid4 = ~np.isnan(ts.rolling_alpha)
        ax4.fill_between(ts.dates[valid4],
                         ts.rolling_alpha[valid4], 0,
                         where=ts.rolling_alpha[valid4] >= 0, alpha=0.3, color='green')
        ax4.fill_between(ts.dates[valid4],
                         ts.rolling_alpha[valid4], 0,
                         where=ts.rolling_alpha[valid4] < 0,  alpha=0.3, color='red')
        ax4.plot(ts.dates[valid4], ts.rolling_alpha[valid4], color='steelblue', linewidth=1.2)
        ax4.axhline(0, color='black', linewidth=0.5)
        ax4.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax4.set_title('Rolling Alpha (vs Benchmark)')

    # 5. Scalar metrics table
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.axis('off')
    m = ts.scalar_metrics
    rows = [
        ['Ann. Return',    f"{m.get('annualized_return',0)*100:.2f}%"],
        ['Volatility',     f"{m.get('volatility',0)*100:.2f}%"],
        ['Sharpe',         f"{m.get('sharpe',0):.3f}"],
        ['Sortino',        f"{m.get('sortino',0):.3f}"],
        ['Calmar',         f"{m.get('calmar',0):.3f}"],
        ['Max Drawdown',   f"{m.get('max_drawdown',0)*100:.2f}%"],
        ['DD Duration',    f"{int(m.get('max_dd_duration',0))} qtrs"],
        ['Alpha (ann.)',   f"{m.get('alpha',0)*100:.2f}%"],
        ['Beta',           f"{m.get('beta',0):.3f}"],
        ['Info. Ratio',    f"{m.get('information_ratio',0):.3f}"],
        ['Tax Drag',       f"{m.get('effective_tax_drag',0)*100:.3f}%"],
        ['Avg Turnover',   f"{m.get('avg_turnover',0)*100:.1f}%"],
        ['Hit Rate',       f"{m.get('hit_rate',0)*100:.1f}%"],
    ]
    tbl = ax5.table(
        cellText=rows, colLabels=['Metric', 'Value'],
        loc='center', cellLoc='left',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1.0, 1.25)
    ax5.set_title('Key Statistics', pad=15)

    fig.suptitle(title, fontweight='bold', fontsize=13)
    plt.show()


plot_tearsheet(report.tearsheets['ppo'], 'PPO Agent — Full Period Tear Sheet')

In [ ]:
# Side-by-side comparison tear sheet
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
strats_ordered = ['ppo', 'equal_weight', 'hold']
strat_colors   = {'ppo': '#2ca02c', 'equal_weight': '#1f77b4', 'hold': '#ff7f0e'}

# Top row: cumulative returns overlaid
ax_top = fig.add_subplot(2, 1, 1)
for name in strats_ordered:
    ts = report.tearsheets[name]
    ax_top.plot(ts.dates, ts.cumulative_returns,
                color=strat_colors[name], linewidth=1.8, label=name.replace('_', ' ').title())
if report.tearsheets['ppo'].benchmark_cumulative is not None:
    ax_top.plot(report.tearsheets['ppo'].dates,
                report.tearsheets['ppo'].benchmark_cumulative,
                color='gray', linewidth=1.0, linestyle='--', label='Benchmark')
ax_top.axhline(1.0, color='black', linewidth=0.4)
ax_top.set_title('Cumulative Return Comparison — All Strategies', fontweight='bold')
ax_top.legend(ncol=4, fontsize=9)

# Bottom row: per-strategy drawdowns
for ax, name in zip(axes[1], strats_ordered):
    ts = report.tearsheets[name]
    ax.fill_between(ts.dates, ts.drawdown, 0, alpha=0.4, color=strat_colors[name])
    ax.plot(ts.dates, ts.drawdown, color=strat_colors[name], linewidth=1.0)
    ax.axhline(0, color='black', linewidth=0.4)
    m = ts.scalar_metrics
    ax.set_title(
        f"{name.replace('_',' ').title()}\n"
        f"Sharpe={m.get('sharpe',0):.2f}  "
        f"MaxDD={m.get('max_drawdown',0)*100:.1f}%",
        fontsize=9
    )
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

# Hide the top-left 3 axes from original grid (we used a 2×1 subplot for top)
for ax in axes[0]:
    ax.set_visible(False)

plt.suptitle('Walk-Forward Backtest — Strategy Comparison', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Walk-Forward Fold Analysis

Simulate fold-level results to show how performance varies across expanding windows.

In [ ]:
# Simulate per-fold metrics by splitting the return series into 3 folds
fold_data: dict[str, list[dict]] = {'ppo': [], 'equal_weight': [], 'hold': []}
fold_dates = []

for fold_i in range(N_FOLDS):
    s = fold_i * FOLD_LEN
    e = s + FOLD_LEN
    fold_dates.append(DATES[s: e])

    for strat_name, ret in [('ppo', r_ppo), ('equal_weight', r_ew), ('hold', r_hold)]:
        fold_r  = ret[s: e]
        fold_bm = r_bm[s: e]
        m = mc.full_metrics(fold_r, fold_bm)
        m['fold_idx'] = float(fold_i)
        fold_data[strat_name].append(m)

# Build a report with fold data injected
from backtest.report import BacktestReport
report_with_folds = BacktestReport(
    strategies=['ppo', 'equal_weight', 'hold'],
    fold_metrics=fold_data,
    aggregate_metrics={
        'ppo':          mc.full_metrics(r_ppo,  r_bm, tc_ppo,  tv_ppo),
        'equal_weight': mc.full_metrics(r_ew,   r_bm, tc_ew,   tv_ew),
        'hold':         mc.full_metrics(r_hold, r_bm, tc_hold, tv_hold),
    },
    tearsheets=report.tearsheets,
    benchmark_name='SPY',
    n_folds=N_FOLDS,
)

print('Per-fold Sharpe:')
print(report_with_folds.fold_summary('sharpe').round(3).to_string())
print()
print('Per-fold Annualized Return:')
print(report_with_folds.fold_summary('annualized_return').applymap(lambda x: f'{x*100:.2f}%').to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric, ylabel in zip(axes,
    ['sharpe', 'annualized_return', 'max_drawdown'],
    ['Sharpe Ratio', 'Ann. Return (%)', 'Max Drawdown (%)']):

    fold_df = report_with_folds.fold_summary(metric)
    x       = np.arange(N_FOLDS)
    w       = 0.25

    for i, (name, color) in enumerate([('ppo','#2ca02c'),('equal_weight','#1f77b4'),('hold','#ff7f0e')]):
        vals = fold_df[name].values if name in fold_df.columns else np.zeros(N_FOLDS)
        if metric in ('annualized_return',):
            vals = vals * 100
        elif metric == 'max_drawdown':
            vals = vals * 100
        ax.bar(x + (i - 1) * w, vals, w, label=name.replace('_',' ').title(),
               color=color, alpha=0.85)

    ax.axhline(0, color='black', linewidth=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {i}' for i in range(N_FOLDS)])
    ax.set_title(ylabel)
    ax.legend(fontsize=7)

plt.suptitle('Per-Fold Performance Comparison', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Tax-Drag Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gross vs after-tax cumulative returns
ax1 = axes[0]
for name, ret, tc_arr, color in [
    ('PPO (gross)',       r_ppo,  tc_ppo,  '#2ca02c'),
    ('PPO (after-tax)',   r_ppo - tc_ppo,  None, '#1a7a1a'),
    ('EW (gross)',        r_ew,   tc_ew,   '#1f77b4'),
    ('EW (after-tax)',    r_ew  - tc_ew,   None, '#0d4a7a'),
    ('Hold (no tax)',     r_hold, None,    '#ff7f0e'),
]:
    cum = MetricsCalculator.cumulative_return_series(ret)
    ax1.plot(DATES, cum, color=color, linewidth=1.5,
             linestyle='--' if 'after-tax' in name else '-',
             label=name)

ax1.axhline(1.0, color='black', linewidth=0.4)
ax1.set_title('Gross vs After-Tax Cumulative Returns')
ax1.legend(fontsize=7, ncol=1)

# Cumulative tax costs over time
ax2 = axes[1]
ax2.stackplot(
    DATES,
    np.cumsum(tc_ppo),
    np.cumsum(tc_ew),
    labels=['PPO cumulative tax', 'EqualWeight cumulative tax'],
    colors=['#2ca02c', '#1f77b4'],
    alpha=0.6,
)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=2))
ax2.set_title('Cumulative Tax Cost (as % of portfolio)')
ax2.legend(fontsize=8)

# Annotate effective tax drag
for name, ret, tc_arr in [('PPO', r_ppo, tc_ppo), ('EW', r_ew, tc_ew)]:
    drag = mc.full_metrics(ret, tc_costs=tc_arr).get('effective_tax_drag', 0.0)
    ax2.annotate(f'{name} drag: {drag*100:.3f}%/yr',
                 xy=(DATES[-1], np.cumsum(tc_arr)[-1]),
                 xytext=(-80, 10), textcoords='offset points',
                 fontsize=8, arrowprops=dict(arrowstyle='->', color='black'))

plt.suptitle('Tax-Drag Analysis — Gross vs After-Tax Performance', fontweight='bold')
plt.tight_layout()
plt.show()

print('Effective annual tax drag:')
for name, ret, tc_arr in [('PPO', r_ppo, tc_ppo), ('EqualWeight', r_ew, tc_ew), ('Hold', r_hold, tc_hold)]:
    m_full = mc.full_metrics(ret, tc_costs=tc_arr)
    gross  = m_full.get('annualized_return', 0) * 100
    drag   = m_full.get('effective_tax_drag', 0) * 100
    after  = gross - drag
    print(f'  {name:<14}: gross={gross:.2f}%, tax drag={drag:.3f}%/yr, after-tax={after:.2f}%')

## 7. Exporting Results

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path  = os.path.join(tmpdir, 'backtest_summary.csv')
    json_path = os.path.join(tmpdir, 'backtest_report.json')

    report.to_csv(csv_path)
    report.to_json(json_path)

    df_loaded = pd.read_csv(csv_path, index_col=0)
    import json
    with open(json_path) as f:
        j_loaded = json.load(f)

print('CSV export:')
print(df_loaded[['annualized_return','sharpe','sortino','max_drawdown']].round(4).to_string())
print()
print('JSON export (top-level keys):', list(j_loaded.keys()))
print('Strategies in JSON:', j_loaded['strategies'])

# TearsheetData as DataFrame
ts_df = report.tearsheets['ppo'].to_dataframe()
print(f'\nTearsheet DataFrame shape: {ts_df.shape}')
print(ts_df.head(3).round(4).to_string())

## 8. GPU Acceleration Summary

In [ ]:
try:
    import cupy as cp
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    print(f'GPU: {gpu_name}')
    mc_gpu = MetricsCalculator(BacktestConfig(use_gpu=True))

    S_large = 50
    T_large = 200
    big_mat = np.random.randn(T_large, S_large) * 0.03 + 0.01

    # CPU timing
    mc_cpu = MetricsCalculator(BacktestConfig(use_gpu=False))
    t0 = time.perf_counter()
    for _ in range(5): mc_cpu.batch_rolling_sharpe(big_mat)
    cpu_ms = (time.perf_counter() - t0) / 5 * 1000

    # GPU timing
    t0 = time.perf_counter()
    for _ in range(5): mc_gpu.batch_rolling_sharpe(big_mat)
    gpu_ms = (time.perf_counter() - t0) / 5 * 1000

    print(f'batch_rolling_sharpe ({T_large}×{S_large}):')
    print(f'  CPU: {cpu_ms:.1f} ms')
    print(f'  GPU: {gpu_ms:.1f} ms  ({cpu_ms/gpu_ms:.1f}× speedup)')

except ImportError:
    print('CuPy not installed — showing CPU timing only')
    S_large = 50; T_large = 200
    big_mat = np.random.randn(T_large, S_large) * 0.03 + 0.01
    t0 = time.perf_counter()
    for _ in range(5): mc.batch_rolling_sharpe(big_mat)
    cpu_ms = (time.perf_counter() - t0) / 5 * 1000
    print(f'  CPU batch_rolling_sharpe ({T_large}×{S_large}): {cpu_ms:.1f} ms')
    print('  Install CuPy for GPU acceleration: pip install cupy-cuda12x')

print()
print('GPU-accelerated operations in the backtest module:')
rows = [
    ('batch_rolling_sharpe',     'CuPy prefix-sum rolling Sharpe',   '8–15×'),
    ('_rolling_sharpe_gpu',      'CuPy prefix-sum per strategy',     '5–10×'),
    ('_rolling_vol_gpu',         'CuPy prefix-sum rolling vol',      '5–10×'),
    ('_rolling_max_drawdown_gpu','CuPy max-accumulate per window',   '4–8×'),
    ('_rolling_alpha_gpu',       'CuPy batched rolling OLS (2×2)',   '6–12×'),
]
print(f'{"Function":<35} {"Method":<38} {"Expected Speedup"}')
print('-' * 88)
for r in rows:
    print(f'{r[0]:<35} {r[1]:<38} {r[2]}')

## Project Complete — All Modules

The full **QuantAgent-RL** system is now implemented:

| Module | Description |
|--------|-------------|
| `data/` | GPU-accelerated ingestion, feature engineering, walk-forward splits |
| `forecasting/` | GARCH vol, HMM regimes, Fama-French factor betas |
| `agents/` | LangGraph multi-agent LLM pipeline → MarketBrief → sentence embedding |
| `rl/` | Gymnasium PortfolioEnv, differential Sharpe reward, PPO agent |
| `backtest/` | Walk-forward metrics, tear sheets, strategy comparison, export |